In [1]:
# Imports
from pert import Pert, plot_gantt_chart, plot_resource_utilization, plot_location_utilization, plot_equipment_utilization

In [ ]:
#!/usr/bin/env python3
"""
Run RCPSP end-to-end:
- Load outage JSON
- Compute CPM
- Compute resource-constrained schedule
- Print summary
- Export CSV
- Generate plots (HTML)
"""

import logging
from datetime import datetime
from pert import Pert, plot_gantt_chart, plot_resource_utilization, plot_location_utilization

# Configure logging in the runner (avoid setting basicConfig inside the module)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

json_path = "example_10.json"
sgs = "max_use_res_ranked"
gantt_file = "gantt.html"
csv_file: str = "schedule.csv"

resource_type: str = "MECHANIC"
resource_plot_file: str = "mechanics.html"
location_id: str = "LOC_REACTOR_CAVITY"
location_plot_file: str = "cavity.html"


# 1) Load data & build schedule graph
pert = Pert.from_json_file(json_path)

# 1.1) debug situations with schedule
pert.debug_connectivity_and_es()
pert.debug_candidates_and_capacity(hours_ahead=48)

pert.generateInfo()
logging.info(f"CPM Duration: {pert.getProjectDuration():.1f} hours")

# 2) Run RCPSP with capacity-aware selection
logging.info(f"Scheduling strategy: {sgs}")
results = pert.calculateScheduleWithResources(sgs=sgs, max_time_hours=24*7)
pert.print_chain_sets_summary()
pert.explain_idle_on_chain()

print("================")
pert.explain_idle_on_chain_detailed()

logging.info(f"RCPSP Completed: {results['n_completed']}/{results['n_activities']}")
logging.info(f"Actual Duration: {results['scheduled_duration']:.1f} hours")
logging.info(f"Total Delay: {results['delay_hours']:.1f} hours")

# 2.1) Make interactive plots: Plotly static DAG
"""
pert.plot_activity_dag(
    filename="dag_plotly.html",
    library="plotly",
    highlight="both",
    layer_by="es",
    include_augmented_edges=True) 

pert.plot_activity_dag(
    filename="dag_precedence_plotly.html",
    library="plotly",
    highlight="both",
    layer_by="es",
    include_augmented_edges=False,      # turn off augmented edges for clarity
    show_unscheduled=False,             # hide nodes without start/end
    max_nodes=0                         # keep all scheduled nodes
)"""

pert.plot_activity_dag(
    filename="dag_augmented_plotly.html",
    library="plotly",
    highlight="constrained",            # emphasize constrained chain
    layer_by="topo",                    # more stable ranks than ES
    include_augmented_edges=True,
    show_unscheduled=False,
    show_edge_arrows=True
)

# 3) Print a summary (do this FIRST so you see output even if export/plots fail)
try:
    pert.print_schedule_summary()
except Exception as e:
    logging.error(f"Error while printing schedule summary: {e}")

# 4) Export CSV (guarded)
try:
    pert.export_schedule_to_csv(csv_file)
    logging.info(f"CSV exported: {csv_file}")
except Exception as e:
    logging.error(f"CSV export failed: {e}")

# 5) Generate plots (guarded)
try:
    plot_gantt_chart(pert, filename=gantt_file, show_delays=True)
    logging.info(f"Gantt chart: {gantt_file}")
except Exception as e:
    logging.error(f"Gantt chart failed: {e}")

for resource_type in pert.resource_pool.get_all_skills():
    resource_plot_file = str(resource_type) + ".html"
    try:
        plot_resource_utilization(pert, resource_type, filename=resource_plot_file)
        logging.info(f"Resource utilization ({resource_type}): {resource_plot_file}")
    except Exception as e:
        logging.error(f"Resource utilization plot failed: {e}")

for location_id in pert.location_pool.get_all_location_ids():
    location_plot_file = str(location_id) + ".html"
    try:
        plot_location_utilization(pert, location_id, filename=location_plot_file)
        logging.info(f"Location utilization ({location_id}): {location_plot_file}")
    except Exception as e:
        logging.error(f"Location utilization plot failed: {e}")

for equipment_id in pert.equipment_pool.get_all_equipment_ids():
    equipment_plot_file = str(equipment_id) + ".html"
    try:
        plot_equipment_utilization(pert, equipment_id, filename=equipment_plot_file)
        logging.info(f"Location utilization ({equipment_id}): {equipment_plot_file}")
    except Exception as e:
        logging.error(f"Location utilization plot failed: {e}")

# 6) Extra diagnostics (optional): show a few steps from the scheduling log
if hasattr(pert, "schedule_log") and pert.schedule_log:
    head = pert.schedule_log[:5]
    logging.info("First scheduling steps (diagnostic):")
    for step in head:
        logging.info(
            "t=%s candidates=%s selected=%s ongoing=%s completed=%s",
            step['time'].strftime('%Y-%m-%d %H:%M'),
            step['candidates'],
            step['selected'],
            step['ongoing'],
            step['completed']
        )
else:
    logging.info("No schedule_log available or empty.")


INFO:root:CPM Duration: 40.0 hours
INFO:root:Scheduling strategy: max_use_res_ranked
INFO:root:Bootstrapped START at 2025-09-01 00:00
INFO:root:Starting RCPSP scheduling at 2025-09-01 00:00:00
INFO:root:Total activities to schedule: 12
INFO:root:Strategy: max_use_res_ranked
INFO:root:RCPSP Completed: 12/12
INFO:root:Actual Duration: 86.0 hours
INFO:root:Total Delay: 55.0 hours


=== Connectivity & ES debug ===
START successors: ['C101']
  C101 ES=1.0h, EF=7.0h
=== End connectivity & ES ===
=== Candidates & capacity debug ===
[2025-09-01 00:00] candidates: []
  Note: C101 abs ES is 2025-09-01 01:00
[2025-09-01 01:00] candidates: []
  Note: C101 abs ES is 2025-09-01 01:00
[2025-09-01 02:00] candidates: []
  Note: C101 abs ES is 2025-09-01 01:00
[2025-09-01 03:00] candidates: []
  Note: C101 abs ES is 2025-09-01 01:00
[2025-09-01 04:00] candidates: []
  Note: C101 abs ES is 2025-09-01 01:00
[2025-09-01 05:00] candidates: []
  Note: C101 abs ES is 2025-09-01 01:00
[2025-09-01 06:00] candidates: []
  Note: C101 abs ES is 2025-09-01 01:00
[2025-09-01 07:00] candidates: []
  Note: C101 abs ES is 2025-09-01 01:00
[2025-09-01 08:00] candidates: []
  Note: C101 abs ES is 2025-09-01 01:00
[2025-09-01 09:00] candidates: []
  Note: C101 abs ES is 2025-09-01 01:00
[2025-09-01 10:00] candidates: []
  Note: C101 abs ES is 2025-09-01 01:00
[2025-09-01 11:00] candidates: []
  N

INFO:numexpr.utils:NumExpr defaulting to 12 threads.
INFO:root:Schedule exported to schedule.csv
INFO:root:CSV exported: schedule.csv
INFO:root:Gantt chart: gantt.html
INFO:root:Resource utilization (MECHANIC): MECHANIC.html
INFO:root:Resource utilization (HP_TECH): HP_TECH.html
INFO:root:Location utilization (LOC_CONTAINMENT): LOC_CONTAINMENT.html
INFO:root:Location utilization (EQ_POLAR_CRANE): EQ_POLAR_CRANE.html
INFO:root:First scheduling steps (diagnostic):
INFO:root:t=2025-09-01 01:00 candidates=['C101'] selected=[] ongoing=[] completed=['START']
INFO:root:t=2025-09-01 02:00 candidates=['C101'] selected=[] ongoing=[] completed=['START']
INFO:root:t=2025-09-01 03:00 candidates=['C101'] selected=[] ongoing=[] completed=['START']
INFO:root:t=2025-09-01 04:00 candidates=['C101'] selected=[] ongoing=[] completed=['START']
INFO:root:t=2025-09-01 05:00 candidates=['C101'] selected=[] ongoing=[] completed=['START']


Gantt chart saved to gantt.html
Resource utilization chart saved to MECHANIC.html
Resource utilization chart saved to HP_TECH.html
Location utilization chart saved to LOC_CONTAINMENT.html
[EQ_POLAR_CRANE] Peak in-use: 1 at 2025-09-02 14:00
  Consumers at peak: ['C102']
Equipment utilization chart saved to EQ_POLAR_CRANE.html
